# Cloud Audit Logs — Hands-On Demo

Companion notebook to the **Cloud Audit Logs Handbook**.

This notebook enables Data Access audit logs for BigQuery, generates a few real events (a table creation, a successful query, and an IAM change), then reads those audit log entries back programmatically using `google-cloud-logging` — printing who did what, when, and whether it was allowed or denied.

If you completed the companion **Data Catalog / policy tags** demo notebook first, this reuses the same `hr_demo.employees` table so you can watch a policy-tag denial appear as an audit log entry.

Run the cells top to bottom. Update the variables in the **Configuration** cell first.


## 0. Setup: install & authenticate


In [ ]:
!pip install --quiet google-cloud-logging google-cloud-bigquery


In [ ]:
try:
    from google.colab import auth
    auth.authenticate_user()
    print('Authenticated via Colab.')
except ImportError:
    print('Not running in Colab -- make sure you have run `gcloud auth application-default login` locally.')


## 1. Configuration — edit before running


In [ ]:
PROJECT_ID = "himanshugcpproject"   # <-- your GCP project
DATASET_ID = "audit_demo"
TABLE_ID = "employees"

print(f"Project: {PROJECT_ID} | Dataset: {DATASET_ID}")


## 2. Enable Data Access audit logs for BigQuery

This patches the project's IAM policy to turn on ADMIN_READ, DATA_READ, and DATA_WRITE audit logging for the BigQuery service. Admin Activity logs need no setup — they are always on.


In [ ]:
from google.cloud import resourcemanager_v3
import google.auth
import google.auth.transport.requests
import requests

creds, _ = google.auth.default()
creds.refresh(google.auth.transport.requests.Request())
headers = {
    "Authorization": f"Bearer {creds.token}",
    "Content-Type": "application/json; charset=utf-8",
}

# 1. Get the current IAM policy
get_url = f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}:getIamPolicy"
policy = requests.post(get_url, headers=headers, json={"options": {"requestedPolicyVersion": 3}}).json()

# 2. Add / merge the BigQuery audit config
audit_configs = policy.get("auditConfigs", [])
bq_config = next((c for c in audit_configs if c.get("service") == "bigquery.googleapis.com"), None)
if bq_config is None:
    bq_config = {"service": "bigquery.googleapis.com", "auditLogConfigs": []}
    audit_configs.append(bq_config)
existing_types = {c["logType"] for c in bq_config["auditLogConfigs"]}
for log_type in ["ADMIN_READ", "DATA_READ", "DATA_WRITE"]:
    if log_type not in existing_types:
        bq_config["auditLogConfigs"].append({"logType": log_type})
policy["auditConfigs"] = audit_configs

# 3. Set the updated policy
set_url = f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}:setIamPolicy"
resp = requests.post(set_url, headers=headers, json={"policy": policy})
print(resp.status_code)
print("BigQuery Data Access audit logging enabled (ADMIN_READ, DATA_READ, DATA_WRITE).")


## 3. Generate a few real events


In [ ]:
from google.cloud import bigquery

bq_client = bigquery.Client(project=PROJECT_ID)

# Admin Activity event: create a dataset
dataset_ref = bigquery.DatasetReference(PROJECT_ID, DATASET_ID)
dataset = bigquery.Dataset(dataset_ref)
dataset.location = "us"
dataset = bq_client.create_dataset(dataset, exists_ok=True)
print(f"Dataset ready: {dataset.dataset_id}")

# Data Access event: run a query (creates the table + reads it back)
query = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}` AS
SELECT 1 AS employee_id, 'Riya Sharma' AS full_name
"""
bq_client.query(query).result()

read_query = f"SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`"
rows = list(bq_client.query(read_query).result())
print(f"Query ran successfully, {len(rows)} row(s) returned.")
print("Give Cloud Logging a minute or two to ingest these events before querying them below.")


## 4. Read the audit log entries back

Uses `google-cloud-logging` to search for Data Access and Admin Activity entries generated by this notebook's own actions, and prints the key fields: who, what, when, and the result.


In [ ]:
import time
from google.cloud import logging as cloud_logging

time.sleep(30)  # give the logging pipeline a little time to catch up

log_client = cloud_logging.Client(project=PROJECT_ID)

filter_str = (
    f'logName=("projects/{PROJECT_ID}/logs/cloudaudit.googleapis.com%2Fdata_access" OR '
    f'"projects/{PROJECT_ID}/logs/cloudaudit.googleapis.com%2Factivity")'
)

entries = list(log_client.list_entries(filter_=filter_str, order_by=cloud_logging.DESCENDING, max_results=20))
print(f"Found {len(entries)} recent audit log entries:\n")

for entry in entries:
    payload = entry.payload or {}
    auth_info = payload.get("authenticationInfo", {})
    who = auth_info.get("principalEmail", "unknown")
    what = payload.get("methodName", "unknown")
    status = payload.get("status", {})
    result = "DENIED" if status.get("code") else "ALLOWED"
    print(f"{entry.timestamp} | {who:35s} | {what:45s} | {result}")


## 5. (Optional) Create a sink to export these logs to BigQuery

For long-term retention and SQL analysis. Requires the Logging Admin / Logs Configuration Writer role. Note: after creating the sink you must grant the printed service account `roles/bigquery.dataEditor` on the destination dataset, or exports will silently fail — see the handbook, Section 7b.


In [ ]:
from google.cloud.logging_v2.services.config_service_v2 import ConfigServiceV2Client
from google.cloud.logging_v2.types import LogSink

CREATE_SINK = False  # flip to True to actually create the sink

if CREATE_SINK:
    config_client = ConfigServiceV2Client()
    sink = LogSink(
        name="audit-logs-to-bq-notebook",
        destination=f"bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{DATASET_ID}",
        filter='logName:"cloudaudit.googleapis.com"',
    )
    parent = f"projects/{PROJECT_ID}"
    created_sink = config_client.create_sink(parent=parent, sink=sink)
    print("Sink created. Writer identity (grant this BigQuery Data Editor on the dataset):")
    print(created_sink.writer_identity)
else:
    print("Skipped. Set CREATE_SINK = True and re-run to create the export sink.")


## 6. Cleanup

Removes the demo dataset/table created by this notebook.


In [ ]:
CONFIRM_CLEANUP = False  # flip to True to actually delete resources

if CONFIRM_CLEANUP:
    bq_client.delete_dataset(dataset_ref, delete_contents=True, not_found_ok=True)
    print("Dataset deleted.")
else:
    print("Skipped. Set CONFIRM_CLEANUP = True and re-run this cell to clean up.")
